In [2]:
# ── Reproducibility Header ────────────────────────────────────────────
# Every notebook in IIT414W starts here. Do not skip this block.

import sys, os, random
import numpy as np
import pandas as pd
import warnings
import fastf1

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

cache_path = os.path.join(os.getcwd(), '..', '..', 'labs', 'lab_01', 'data', 'fastf1_cache')
os.makedirs(cache_path, exist_ok=True)
fastf1.Cache.enable_cache(cache_path)

print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'fastf1  : {fastf1.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.10.20
NumPy   : 1.26.4
fastf1  : 3.3.9
Seed    : 414


# Lab 2: Feature Engineering & Simple Modeling

**Objective:** Improve upon the Lab 1 baseline (accuracy ~74%, F1 ~0.74) by engineering domain-specific features without using future information (leakage).

In [3]:
# Load processed data from Lab 1
# We look for the file relative to this notebook location
data_path = os.path.join('..', 'lab_01', 'data', 'processed', 'results_2022_2024.csv')

if os.path.exists(data_path):
    df = pd.read_csv(data_path, parse_dates=['date'])
    # Ensure booleans are correct
    df['top10'] = df['top10'].astype(bool)
    df['finished'] = df['finished'].astype(bool)
    # Ensure numeric
    df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
    df['position'] = pd.to_numeric(df['position'], errors='coerce')
else:
    raise FileNotFoundError(f"Data file not found at {data_path}. Please run Lab 1 baseline.ipynb first.")

print(f"Loaded {len(df)} rows.")
df.head()

Loaded 1359 rows.


,season,round,race,date,circuit,driverId,driver,constructor,position,positionText,grid,laps,status,points,top10,finished
0,2022,1,Bahrain Grand Prix,2022-03-20,bahrain,leclerc,Charles Leclerc,Ferrari,1,1,1,57,Finished,26,True,True
1,2022,1,Bahrain Grand Prix,2022-03-20,bahrain,sainz,Carlos Sainz,Ferrari,2,2,3,57,Finished,18,True,True
2,2022,1,Bahrain Grand Prix,2022-03-20,bahrain,hamilton,Lewis Hamilton,Mercedes,3,3,5,57,Finished,15,True,True
3,2022,1,Bahrain Grand Prix,2022-03-20,bahrain,russell,George Russell,Mercedes,4,4,9,57,Finished,12,True,True
4,2022,1,Bahrain Grand Prix,2022-03-20,bahrain,kevin_magnussen,Kevin Magnussen,Haas F1 Team,5,5,7,57,Finished,10,True,True


## Feature Engineering

We will add three new features based on F1 domain knowledge:
1.  **Lag Feature (`prev_race_position`):** The driver's finishing position in the previous race. Justification: Recent performance is a strong predictor of current form.
2.  **Rolling Aggregate (`avg_pos_last_3`):** Average finishing position over the last 3 races. Justification: Smoothes out single-race anomalies (crashes, mechanical failures) to capture underlying pace.
3.  **Categorical Encoding (`constructor_tier`):** Tiers based on 2021 Championship standings (prior knowledge). Justification: Car performance dominates F1 results; specific teams consistently outperform others.

### Leakage Guard Checklist

We apply the shift(1) pattern to ensure no current-race information is used.

In [4]:
# 1. Constructor Tier (Categorical)
# Based on 2021 Constructors' Championship (valid prior knowledge for 2022 season start)
# Tier 1: Mercedes, Red Bull, Ferrari, McLaren (Top 4)
# Tier 2: Alpine, AlphaTauri, Aston Martin, Williams (Mid/Back)
# Tier 3: Alfa Romeo, Haas (Back)

def get_tier(constructor_name):
    if constructor_name in ['Mercedes', 'Red Bull', 'Ferrari', 'McLaren']:
        return 1
    elif constructor_name in ['Alpine F1 Team', 'AlphaTauri', 'Aston Martin', 'Williams']:
        return 2
    else:
        return 3

df['constructor_tier'] = df['constructor'].apply(get_tier)
print("Constructor Tiers created.")

# 2. Lag Feature: Previous Race Position
# Group by driver and shift 1
df['prev_race_position'] = df.groupby('driver')['position'].shift(1)

# 3. Rolling Aggregate: Average Position Last 3 Races
# We must shift *before* rolling or shift *after* rolling. 
# rolling(3).mean() includes currents. So we shift(1) first to get "previous 3".
# Or verify pattern: shift(1) gives prev. rolling on that gives avg of prev.
df['avg_pos_last_3'] = df.groupby('driver')['position'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# Fill NaNs specifically for the first races
# For lag/rolling, missing values (start of season/career) can be filled with grid position or a default.
# We will fill with `grid` as a proxy for expectation, or median.
# Using grid is reasonable: if we don't know past form, we trust the qualy pace.
df['prev_race_position'] = df['prev_race_position'].fillna(df['grid'])
df['avg_pos_last_3'] = df['avg_pos_last_3'].fillna(df['grid'])

# Check for leakage
# Feature values should NOT be identical to 'position'
print("Leakage Check - Correlation with Target (position):")
print(df[['position', 'prev_race_position', 'avg_pos_last_3']].corr())

# Sample check
df[['date', 'driver', 'grid', 'position', 'prev_race_position', 'avg_pos_last_3']].head(10)

Constructor Tiers created.
Leakage Check - Correlation with Target (position):
                    position  prev_race_position  avg_pos_last_3
position            1.000000            0.464126        0.551418
prev_race_position  0.464126            1.000000        0.807345
avg_pos_last_3      0.551418            0.807345        1.000000


,date,driver,grid,position,prev_race_position,avg_pos_last_3
0,2022-03-20,Charles Leclerc,1,1,1.0,1.0
1,2022-03-20,Carlos Sainz,3,2,3.0,3.0
2,2022-03-20,Lewis Hamilton,5,3,5.0,5.0
3,2022-03-20,George Russell,9,4,9.0,9.0
4,2022-03-20,Kevin Magnussen,7,5,7.0,7.0
5,2022-03-20,Valtteri Bottas,6,6,6.0,6.0
6,2022-03-20,Esteban Ocon,11,7,11.0,11.0
7,2022-03-20,Yuki Tsunoda,16,8,16.0,16.0
8,2022-03-20,Fernando Alonso,8,9,8.0,8.0
9,2022-03-20,Guanyu Zhou,15,10,15.0,15.0


## Modeling

We use a Decision Tree Classifier as requested in the plan.
**Features:** `grid`, `constructor_tier`, `prev_race_position`, `avg_pos_last_3`.
**Target:** `top10`

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, confusion_matrix

# Simple preparation
# We need to drop rows where grid is NaN (though Lab 1 handles this, we double check)
model_df = df.dropna(subset=['grid', 'prev_race_position', 'avg_pos_last_3', 'constructor_tier']).copy()

# Split (Temporal)
train = model_df[model_df['season'] == 2022]
val = model_df[model_df['season'] == 2023]
# Test would be 2024, but we focus on val for now.

X_features = ['grid', 'constructor_tier', 'prev_race_position', 'avg_pos_last_3']
y_target = 'top10'

X_train = train[X_features]
y_train = train[y_target].astype(int)

X_val = val[X_features]
y_val = val[y_target].astype(int)

# Initialize and Train
# max_depth=5 is a reasonable starting point for a simple tree to avoid overfitting
model = DecisionTreeClassifier(random_state=RANDOM_SEED, max_depth=5)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_val)

# Evaluate
print("=== Lab 2 Decision Tree Results (Validation 2023) ===")
print(classification_report(y_val, y_pred, target_names=['Non-Top10', 'Top10']))

f1 = f1_score(y_val, y_pred)
print(f"F1 Score: {f1:.4f}")

=== Lab 2 Decision Tree Results (Validation 2023) ===
              precision    recall  f1-score   support

   Non-Top10       0.74      0.80      0.77       220
       Top10       0.78      0.73      0.75       220

    accuracy                           0.76       440
   macro avg       0.76      0.76      0.76       440
weighted avg       0.76      0.76      0.76       440

F1 Score: 0.7529


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.dummy import DummyClassifier

# 1. Majority Class Baseline (Lab 1)
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_val)
y_prob_dummy = dummy.predict_proba(X_val)[:, 1]

# 2. Domain Heuristic Baseline (Lab 1)
# Grid <= 10
y_pred_heuristic = (X_val['grid'] <= 10).astype(int)
# Heuristic "prob" is just 1 if grid<=10 else 0 (soft proxy)
y_prob_heuristic = y_pred_heuristic

# 3. Lab 2 Model (Decision Tree)
y_pred_dt = model.predict(X_val)
y_prob_dt = model.predict_proba(X_val)[:, 1]

def get_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_prob) if y_prob is not None else 0.5
    }

metrics = {
    'Majority class (Lab 1)': get_metrics(y_val, y_pred_dummy, y_prob_dummy),
    'Domain heuristic (Lab 1)': get_metrics(y_val, y_pred_heuristic, y_prob_heuristic),
    'Lab 2 model (Decision Tree)': get_metrics(y_val, y_pred_dt, y_prob_dt)
}

print(f"{'Model / Baseline':<30} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1':<10} | {'ROC-AUC':<10}")
print("-" * 95)
for name, m in metrics.items():
    print(f"{name:<30} | {m['Accuracy']:.4f}     | {m['Precision']:.4f}      | {m['Recall']:.4f}     | {m['F1']:.4f}     | {m['ROC-AUC']:.4f}")

Model / Baseline               | Accuracy   | Precision  | Recall     | F1         | ROC-AUC   
-----------------------------------------------------------------------------------------------
Majority class (Lab 1)         | 0.5000     | 0.0000      | 0.0000     | 0.0000     | 0.5000
Domain heuristic (Lab 1)       | 0.7295     | 0.7265      | 0.7364     | 0.7314     | 0.7295
Lab 2 model (Decision Tree)    | 0.7614     | 0.7805      | 0.7273     | 0.7529     | 0.7676


## Error Analysis

We examine the failures (False Positives and False Negatives) to understand model limitations.

In [6]:
val['pred_top10'] = y_pred
val['correct'] = val['top10'] == val['pred_top10']
errors = val[~val['correct']].copy()

print(f"Total Errors: {len(errors)} out of {len(val)}")

# View Top 3 Failure Modes
# 1. False Positives (Predicted Top 10, but wasn't)
fp = errors[errors['pred_top10'] == 1]
print("\n--- False Positives (Predicted Top 10, Actual Bottom 10) ---")
print(fp[['race', 'driver', 'grid', 'position', 'finished']].head(5))

# 2. False Negatives (Predicted Bottom 10, Actual Top 10)
fn = errors[errors['pred_top10'] == 0]
print("\n--- False Negatives (Predicted Bottom 10, Actual Top 10) ---")
print(fn[['race', 'driver', 'grid', 'position', 'finished']].head(5))

# Analysis of DNFs in False Positives
fp_dnf = fp[fp['finished'] == False]
print(f"\nFalse Positives that were DNFs: {len(fp_dnf)} ({len(fp_dnf)/len(fp)*100:.1f}%)")

Total Errors: 105 out of 440

--- False Positives (Predicted Top 10, Actual Bottom 10) ---
                         race           driver  grid  position  finished
457        Bahrain Grand Prix     Esteban Ocon     9        18     False
458        Bahrain Grand Prix  Charles Leclerc     3        19     False
474  Saudi Arabian Grand Prix    Oscar Piastri     8        15      True
479  Saudi Arabian Grand Prix     Lance Stroll     5        20     False
491     Australian Grand Prix     Carlos Sainz     5        12      True

--- False Negatives (Predicted Bottom 10, Actual Top 10) ---
                         race           driver  grid  position  finished
445        Bahrain Grand Prix     Lance Stroll     8         6      True
447        Bahrain Grand Prix  Valtteri Bottas    12         8      True
448        Bahrain Grand Prix     Pierre Gasly    20         9      True
449        Bahrain Grand Prix  Alexander Albon    15        10      True
468  Saudi Arabian Grand Prix     Pierre Gas

C:\Users\benja\AppData\Local\Temp\ipykernel_4900\3012079164.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val['pred_top10'] = y_pred
C:\Users\benja\AppData\Local\Temp\ipykernel_4900\3012079164.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val['correct'] = val['top10'] == val['pred_top10']


## Error Analysis Summary using Top-3 Failure Modes

1.  **The "DNF" Surprise (False Positives):** A major source of error is predicting a driver will finish Top 10 because they qualified well (low grid) and have good history, but they crash (DNF). Our model doesn't predict crashes.
    *   *Example:* Leclerc in 2023 Bahrain (Engine failure).
2.  **Recovery Drives (False Negatives):** Drivers starting at the back (high grid) due to penalties or qualifying errors often recover to Top 10 if they are in a top car. Our model over-weights `grid` and might miss the car's inherent speed advantage despite the penalty.
    *   *Example:* Verstappen starting P15 (Saudi Arabia 2023) and finishing P2.
3.  **Mid-Field Chaos:** Cars in the "Tier 2" often swap positions around P10/P11. The margin is razor thin, and small track incidents determine the outcome.

**Next Steps:**
*   Adding "penalty" flag to features could help (separating slow qualifying from penalty drops).
*   Predicting "Probability of Finishing" separately.

## Leakage Guard Checklist

*   [x] **No future info:** Used `.shift(1)` for lag features.
*   [x] **Rolling windows:** Shifted *before/after* rolling to exclude current race.
*   [x] **Target separation:** Target `top10` (from `position`) is kept separate from features.
*   [x] **Validation split:** Strictly temporal (Train 2022, Val 2023).